Shared memory optimization is one of the most important performance upgrades in CUDA programming. Each thread repeatedly reads data from global memory, which is slow.

Shared memory solves this by:
* Loading data once per block
* Reusing it multiple times
* Reducing global memory traffic dramatically
* This can improve performance by 10×–100× depending on workload size.

In [1]:
# Setup PyCUDA (Colab / Linux)
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 31.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 11.3 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp312-cp312-linux_x86_64.whl size=659447 sha256=a2619548f83812dba32d6a95aa801a35d4257374b3837919416d72ab423c0941
  Stored in directory: /root/.cache/pip/wheels/90/2a/71/75ec0cc316cc0ff494bfffa2935e02580129cb7f859a0cfd8f
Successfully built pycuda


In [2]:
#Cek GPU
!nvidia-smi

Thu Jun 11 00:56:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# **Shared Memory Optimization in CUDA (PyCUDA)**


In [ ]:
# Problem: Naive Matrix Multiplication (CPU Baseline)

In [4]:
# We first compare with CPU

import numpy as np
import time

def cpu_matmul(A, B):
    start = time.time()
    C = np.dot(A, B)
    end = time.time()
    return C, end - start

In [ ]:
# GPU Naive Version (Global Memory Only)
# Each thread directly accesses global memory:

In [5]:
# CUDA Kernel (Shared Memory Optimization)

import pycuda.autoinit
import pycuda.driver as cuda
import numpy as np
from pycuda.compiler import SourceModule
import time

mod = SourceModule("""
#define TILE 16

__global__ void matmul_shared(float *A, float *B, float *C, int N)
{
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];

    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;

    float value = 0;

    for (int t = 0; t < (N + TILE - 1) / TILE; t++)
    {
        As[threadIdx.y][threadIdx.x] = A[row * N + (t * TILE + threadIdx.x)];
        Bs[threadIdx.y][threadIdx.x] = B[(t * TILE + threadIdx.y) * N + col];

        __syncthreads();

        for (int k = 0; k < TILE; k++)
        {
            value += As[threadIdx.y][k] * Bs[k][threadIdx.x];
        }

        __syncthreads();
    }

    C[row * N + col] = value;
}
""")

func = mod.get_function("matmul_shared")

In [6]:
# GPU Function (Shared Memory Version)
def gpu_shared(A, B):
    N = A.shape[0]
    C = np.zeros((N, N), dtype=np.float32)

    A_gpu = cuda.mem_alloc(A.nbytes)
    B_gpu = cuda.mem_alloc(B.nbytes)
    C_gpu = cuda.mem_alloc(C.nbytes)

    cuda.memcpy_htod(A_gpu, A)
    cuda.memcpy_htod(B_gpu, B)

    TILE = 16
    block = (TILE, TILE, 1)
    grid = ((N + TILE - 1)//TILE, (N + TILE - 1)//TILE)

    start = cuda.Event()
    end = cuda.Event()

    start.record()

    func(A_gpu, B_gpu, C_gpu, np.int32(N),
         block=block,
         grid=grid)

    end.record()
    end.synchronize()

    cuda.memcpy_dtoh(C, C_gpu)

    gpu_time = start.time_till(end) / 1000
    return C, gpu_time

# **Benchmark CPU vs GPU**

In [7]:
sizes = [64, 128, 256, 512]

cpu_times = []
gpu_times_naive = []
gpu_times_shared = []

for N in sizes:
    A = np.random.rand(N, N).astype(np.float32)
    B = np.random.rand(N, N).astype(np.float32)

    _, t_cpu = cpu_matmul(A, B)

    # shared memory GPU
    _, t_gpu = gpu_shared(A, B)

    cpu_times.append(t_cpu)
    gpu_times_shared.append(t_gpu)

    print(f"N={N} | CPU={t_cpu:.4f}s | GPU Shared={t_gpu:.4f}s")

N=64 | CPU=0.0018s | GPU Shared=0.0005s
N=128 | CPU=0.0034s | GPU Shared=0.0001s
N=256 | CPU=0.0005s | GPU Shared=0.0002s
N=512 | CPU=0.0027s | GPU Shared=0.0007s
